<h1><center><strong></strong></center></h1>
<h1><center><strong>CME466</strong></center></h1>
<h1><center><strong>Design of an Advanced Digital System</strong></center></h1>
<p><center><strong>Department of Electrical and Computer Engineering</strong></center></p>
<p><center><strong>2026 Winter Term</strong></center></p>
<p><center><strong>Last update: March 07, 2026</strong></center></p>

# Introduction to CNN - Part 1
# CIFAR-10 Classification Using CNNs

For tips on running notebooks in Google Colab, see
https://docs.pytorch.org/tutorials/beginner/colab

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

# Import matplotlib for visualization
import matplotlib.pyplot as plt
from pathlib import Path

#ensures that plots are displayed as static images directly within the output cells
%matplotlib inline

In [ ]:
# Set up device agnostic code
# automatically adapts to the available hardware, such as a CPU, NVIDIA GPU (CUDA), or Apple GPU (MPS)
# CUDA is NVIDIA's parallel computing platform and programming model that allows software to use GPU acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
device

Training a Classifier
=====================

Generally, when you have to deal with image, text, audio or video data,
you can use standard python packages that load data into a numpy array.
Then you can convert this array into a `torch.*Tensor`.

-   For images, packages such as Pillow, OpenCV are useful
-   For audio, packages such as scipy and librosa
-   For text, either raw Python or Cython based loading, or NLTK and
    SpaCy are useful

Specifically for vision, we have created a package called `torchvision`,
that has data loaders for common datasets such as ImageNet, CIFAR10,
MNIST, etc. and data transformers for images, viz.,
`torchvision.datasets` and `torch.utils.data.DataLoader`.

This provides a huge convenience and avoids writing boilerplate code.

Here, we will use the CIFAR10 dataset. It has 10  classes:
'airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse',
'ship', 'truck'. The images in CIFAR-10 are of size 3x32x32, i.e.
3-channel color images of 32x32 pixels in size.

![cifar10](https://pytorch.org/tutorials/_static/img/cifar10.png)

Training an image classifier
----------------------------

We will do the following steps in order:

1.  Load and normalize the CIFAR10 training and test datasets using
    `torchvision`
2.  Define a Convolutional Neural Network
3.  Define a loss function
4.  Train the network on the training data
5.  Test the network on the test data

### 1. Load and normalize CIFAR10

Using `torchvision`, it's extremely easy to load CIFAR10.


The output of torchvision datasets are PILImage images of range \[0,
1\]. We transform them to Tensors of normalized range \[-1, 1\].


The CIFAR-10 dataset is a fundamental machine learning dataset containing 60,000 32x32 pixel color images, divided into 10 distinct classes. It consists of 50,000 training images and 10,000 test images.

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
print(f'trainset size: {len(trainset)}')
print(f'testset size: {len(testset)}')
# How many samples are there?
#print(f"Dataloaders: {trainloader, testloader}")
print(f"Length of train dataloader: {len(trainloader)} batches of {batch_size}")
print(f"Length of test dataloader: {len(testloader)} batches of {batch_size}")

In [ ]:
# See classes
class_names = trainset.classes
class_names

In [ ]:
# See first training sample
image, label = trainset[0]
# image, label
print(image.shape)

Let us show some of the training images, for fun.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# helper functions to show an image

def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


In [ ]:
# get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# show images
imshow(torchvision.utils.make_grid(images))
# print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(batch_size)))

2. Define a Convolutional Neural Network
========================================

We will use the LeNet-5 CNN architecture.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5) # in_channels=3, out_channels=6, kernel_size=5, stride=1,padding=0)
        self.pool = nn.MaxPool2d(2, 2) # kernel_size=2, stride=2
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120) # in_features=16 * 5 * 5, out_features=120
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net().to(device)
print(net)

In [ ]:
# Try to import torchinfo, install if it doesn't work
# https://pypi.org/project/torchinfo/
try:
  from torchinfo import summary
except:
  print("[INFO] coudln't find torchinfo, installing it...")
  !pip install -q torchinfo
  from torchinfo import summary
  print("[INFO] Done!")

In [ ]:
from torchinfo import summary
summary(model=net,
        input_size=(4, 3, 32, 32), # Corrected input size for CIFAR10 images
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"])

3. Define a Loss function and optimizer
=======================================

Let\'s use a Classification Cross-Entropy loss and SGD with momentum.


In [ ]:
# Define hyperparameters
learning_rate = 1e-3 # 0.001

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=learning_rate, momentum=0.9)

4. Train the network
====================

This is when things start to get interesting. We simply have to loop
over our data iterator, and feed the inputs to the network and optimize.


In [ ]:
# epoch = 2, start with small, takes time....
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels], not X and y as before
        inputs, labels = data

        # Move inputs and labels to the device
        inputs = inputs.to(device)
        labels = labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

Let\'s quickly save our trained model:


In [ ]:
PATH = './cifar_net.pth'
torch.save(net.state_dict(), PATH)

See [here](https://pytorch.org/docs/stable/notes/serialization.html) for
more details on saving PyTorch models.

5. Test the network on the test data
====================================

We have trained the network for 2 passes over the training dataset. But
we need to check if the network has learnt anything at all.

We will check this by predicting the class label that the neural network
outputs, and checking it against the ground-truth. If the prediction is
correct, we add the sample to the list of correct predictions.


Next, let\'s load back in our saved model (note: saving and re-loading
the model wasn\'t necessary here, we only did it to illustrate how to do
so):


In [ ]:
net = Net()
net.load_state_dict(torch.load(PATH, weights_only=True))

In [ ]:
dataiter = iter(testloader)
images, labels = next(dataiter)

# print images
imshow(torchvision.utils.make_grid(images))
print('GroundTruth: ', ' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))

Okay, now let us see what the neural network thinks these examples above
are:


The outputs are energies for the 10 classes. The higher the energy for a
class, the more the network thinks that the image is of the particular
class. So, let\'s get the index of the highest energy:


In [ ]:
outputs = net(images)
_, predicted = torch.max(outputs, 1)

print('Predicted: ', ' '.join(f'{classes[predicted[j]]:5s}'
                              for j in range(4)))

Let us look at how the network performs on the whole dataset.


In [ ]:
correct = 0
total = 0
net.to(device) # Ensure the model is on the correct device
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

With only 2 epochs, the accuracy is only 55%.

Let us see what classes performed well, what did not...


In [ ]:
# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# again no gradients needed
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

Let us also print the Confusion Matrix

In [ ]:
# Try to import torchmetrics and install if it doesn't work
try:
    import torchmetrics
except:
    print("[INFO] coudln't find torchmetrics, installing it...")
    !pip install -q torchmetrics
    import torchmetrics
    print("[INFO] Done!")

In [ ]:
import matplotlib.pyplot as plt
from torchmetrics import ConfusionMatrix
import seaborn as sn
import pandas as pd

# Assuming 'preds' and 'target' are your accumulated tensors of predictions and true labels
# preds should be an integer tensor of predicted labels (e.g., from torch.argmax(outputs, dim=1))
# target should be an integer tensor of true labels

num_classes = 10 # Set the number of classes
metric = ConfusionMatrix(task="multiclass", num_classes=num_classes)

net.eval() # Set model to evaluation mode
all_preds = []
all_targets = []
with torch.no_grad():
    for X, y in testloader:
        X, y = X.to(device), y.to(device)
        pred_logits = net(X)
        pred_labels = pred_logits.argmax(dim=1)
        all_preds.append(pred_labels)
        all_targets.append(y)

# Concatenate all predictions and targets
y_pred = torch.cat(all_preds)
y_true = torch.cat(all_targets)

confmat = ConfusionMatrix(num_classes=num_classes, task='multiclass')
# Ensure tensors are on CPU for torchmetrics if device is CUDA
cm_tensor = confmat(y_pred.cpu(), y_true.cpu())

# Convert to pandas DataFrame for better visualization with seaborn
df_cm = pd.DataFrame(cm_tensor.numpy(), index=range(num_classes), columns=range(num_classes))
plt.figure(figsize = (7,5))
# Use seaborn's heatmap for a nice plot
sn.heatmap(df_cm, annot=True, fmt="d", cmap='Blues')
plt.xlabel("Predicted labels")
plt.ylabel("True labels")
plt.show()

In [ ]:
# helper function to plot results
import random
import numpy as np
import matplotlib.pyplot as plt # Ensure plt is imported here

# Redefine plot_image function to correctly handle image shape and unnormalization
def plot_image(i, predictions_array, true_label, img):
  true_label, img_data = true_label[i], img[i] # Renamed img to img_data to avoid conflict
  plt.grid(False)
  plt.xticks([])
  plt.yticks([])

  # Unnormalize, convert to numpy, and transpose for matplotlib
  img_data = img_data / 2 + 0.5     # unnormalize
  npimg = img_data.numpy()
  plt.imshow(np.transpose(npimg, (1, 2, 0))) # Transpose from (C, H, W) to (H, W, C)

  predicted_label = torch.argmax(predictions_array).item()

  if predicted_label == true_label:
    color = 'blue'
  else:
    color = 'red'

  plt.xlabel("{} {:2.0f}% ({})".format(classes[predicted_label],
                                100*torch.max(predictions_array).item(),
                                classes[true_label]),
                                color=color)

# Redefine plot_value_array function to correct class_names and x-axis range
def plot_value_array(i, predictions_array, true_label):
  true_label = true_label[i]
  plt.grid(False)
  plt.xticks(range(len(classes))) # Use len(classes) for x-ticks
  plt.yticks([])
  thisplot = plt.bar(range(len(classes)), predictions_array, color="#777777") # Use len(classes) for bar range
  plt.ylim([0, 1])
  predicted_label = np.argmax(predictions_array)

  thisplot[predicted_label].set_color('red')
  thisplot[true_label].set_color('blue')



In [ ]:
# Plot the first X test images, their predicted labels, and (true labels).
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 2
num_cols = 2
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))

# Convert dataloader into a list of batches
all_batches = list(testloader)

# Pick a random batch
X_batch, y_batch = random.choice(all_batches)
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# Get model predictions
net.eval()  # Ensure model is in eval mode
with torch.no_grad():
    y_pred = net(X_batch)  # Get raw logits
    y_pred_prob = torch.softmax(y_pred, dim=1)  # Convert logits to probabilities
    y_pred_labels = torch.argmax(y_pred_prob, dim=1)  # Get predicted labels

for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, y_pred_prob[i].cpu(), y_batch.cpu(), X_batch.squeeze().cpu())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, y_pred_prob[i].cpu(), y_batch.cpu())
plt.tight_layout()
plt.show()

# Visualizing Neural Networks using Saliency Maps in PyTorch

In [ ]:
def saliency(img, model):
    #we don't need gradients w.r.t. weights for a trained model
    for param in model.parameters():
        param.requires_grad = False

    #set model in eval mode
    net.eval()

    #we want to calculate gradient of higest score w.r.t. input
    #so set requires_grad to True for input
    img.requires_grad = True

    #forward pass to calculate predictions
    logits = model(img.unsqueeze(0).to(device))

    #get index of class with highest score
    idx = torch.argmax(logits, dim=1)
    score = logits[0][idx]

    #backward pass to get gradients of score predicted class w.r.t. input image
    score.backward()

    #get max along channel axis
    saliency, _ = torch.max(img.grad.data.abs(), dim=0)

    saliency = (saliency - saliency.min())/(saliency.max()-saliency.min())

    #apply inverse transform on image
    with torch.no_grad():
        raw_img = img / 2 + 0.5     # unnormalize
        raw_img = raw_img.numpy()

    #plot image and its saleincy map
    plt.figure(figsize=(10, 10))

    plt.subplot(1, 2, 1)
    plt.imshow(np.transpose(raw_img, (1, 2, 0)))
    plt.xticks([])
    plt.yticks([])

    plt.subplot(1, 2, 2)
    plt.imshow(saliency.cpu().numpy(), cmap='jet', vmin=0.0, vmax=1.0)
    plt.xticks([])
    plt.yticks([])
    plt.show()

    ax = plt.subplot()
    plt.imshow(np.transpose(raw_img, (1, 2, 0)))
    plt.imshow(saliency.cpu().numpy(), cmap='jet', alpha=0.25)
    plt.xticks([])
    plt.yticks([])

Print salienecy maps

In [ ]:
saliency(test_data[2000][0], net)

## Other CNN Models
There are many standard CNN models that we can use directly on our dataset. We can either fully train the model on our own (which takes longer time), use a pretrained model with partial training (this is more common), or use it directly without any modification.

# AlexNet Architecture

https://www.pinecone.io/learn/series/image-search/imagenet/

AlexNet is a groundbreaking 8-layer Convolutional Neural Network (CNN) designed by Alex Krizhevsky, Ilya Sutskever, and Geoffrey Hinton (UofT), which won the 2012 ImageNet ILSVRC challenge with a 15.3% top-5 error rate. It popularized deep learning by using GPUs for faster training, ReLU activation, and Dropout to reduce overfitting, significantly advancing computer vision.

It consists of 5 convolutional layers (some followed by max-pooling layers) and 3 fully connected layers, designed for 227x227 image pixels. The model contains over 60 million parameters and 650,000 neurons.

In [ ]:
import torch
import torchvision.models as models

# Try to import torchinfo, install if it doesn't work
# https://pypi.org/project/torchinfo/
try:
  from torchinfo import summary
except:
  print("[INFO] coudln't find torchinfo, installing it...")
  !pip install -q torchinfo
  from torchinfo import summary
  print("[INFO] Done!")

In [ ]:
#load pretrained alexnet model
model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
summary(model=model,
        input_size=(32, 3, 227, 227),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"])

# Resnet-18 Architecture

ResNet-18 is a convolutional neural network with 18 deep layers, widely used for image classification and computer vision tasks. It has 17 convolutional layers and 1 fully connected layer, designed for 224x224 pixel images.

Part of the Residual Network family, it features skip connections that mitigate vanishing gradients. It has 11.7 million parameters, provides high efficiency, and is often pretrained on ImageNet.

In [ ]:
#load pretrained resnet-18 model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
summary(model=model,
        input_size=(32, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"])

Before using the pre-trained models, one must preprocess the image (resize with right resolution/interpolation, apply inference transforms, rescale the values etc). There is no standard way to do this as it depends on how a given model was trained. It can vary across model families, variants or even weight versions. Using the correct preprocessing method is critical and failing to do so may lead to decreased accuracy or incorrect outputs.

All the necessary information for the inference transforms of each pre-trained model is provided on its weights documentation. To simplify inference, TorchVision bundles the necessary preprocessing transforms into each model weight. These are accessible via the weight.transforms attribute.

We will learn about these models in the next module.

In [ ]:
del dataiter

# Resources
<li><a href="https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html">CIFAR10 Tutorial</a></li>
<li><a href="https://www.learnpytorch.io/">Learn PyTorch for Deep Learning: Zero to Mastery book</a></li>
<li><a href="https://lightning.ai/docs/torchmetrics/stable/classification/accuracy.html">Accuracy</a></li>
<li><a href="https://pytorch.org/vision/main/generated/torchvision.datasets.CIFAR10.html">CIFAR10</a></li>
<li><a href="https://github.com/sunnynevarekar/pytorch-saliency-maps/blob/master/Saliency_maps_in_pytorch.ipynb">Visualizing image-specific class saliency map in classification ConvNets in Pytorch</a></li>
<li><a href="https://medium.datadriveninvestor.com/visualizing-neural-networks-using-saliency-maps-in-pytorch-289d8e244ab4">Visualizing Neural Networks using Saliency Maps in PyTorch</li></a>
<li><a href="https://poloclub.github.io/cnn-explainer/">CNN Explainer</a></li>
<li><a href="https://adamharley.com/nn_vis/cnn/3d.html">3D Visualization of a Convolutional Neuwal Netowrk</a></li>